# Lab 06 — Experiment Tracking and Debugging

Runnable companion to `docs/12_practical_methodology_debugging.md`. We
wire **TensorBoard** into a tiny MNIST training loop, log gradient
norm and weight norm per epoch, then sweep three learning rates so you
can compare them in TensorBoard side by side.

After this lab you should be able to:

- Open TensorBoard on `experiments/lab06/runs/` and see four
  scalar curves per run (train loss, val loss, val acc, grad norm).
- Tell apart "LR too small", "LR sane", "LR too large" from those
  curves alone.
- Add gradient-norm logging to any of the project-01..05 training
  scripts in 6 lines.

Generated by `src/build_lab_06_experiment_tracking.py`.

## Setup

In [1]:
from pathlib import Path as _Path
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device("cpu")

_ROOT = _Path.cwd()
while not (_ROOT / "ROADMAP.md").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
MNIST_ROOT = str(_ROOT / "datasets" / "mnist")
RUNS_ROOT  = _ROOT / "experiments" / "lab06" / "runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print("torch:", torch.__version__, "device:", DEVICE)
print("runs dir:", RUNS_ROOT)

torch: 2.12.0+cu130 device: cpu
runs dir: /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/experiments/lab06/runs


## Small MNIST subset and a small MLP

Same shape as Project 01 but trimmed to keep the lab fast.

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_full = datasets.MNIST(MNIST_ROOT, train=True,  download=True, transform=transform)
val_full   = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(train_full, list(range(4000))), batch_size=64, shuffle=True)
val_loader   = DataLoader(Subset(val_full,   list(range(1000))), batch_size=256)


class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 64), nn.ReLU(inplace=True),
            nn.Linear(64, 10),
        )
    def forward(self, x):
        return self.net(x)

## A tracked training loop

Adds three diagnostics to the bare loop:

- **`SummaryWriter.add_scalar`** for train loss, val loss, val acc.
- **Global gradient L2 norm** logged once per epoch.
- **Mean weight L2 norm** logged once per epoch.

Anything you log in this format shows up in TensorBoard in real time.

In [3]:
def global_grad_norm(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.detach().pow(2).sum().item()
    return total ** 0.5


def global_weight_norm(model):
    total = 0.0
    for p in model.parameters():
        total += p.detach().pow(2).sum().item()
    return total ** 0.5


def train_one_lr(lr: float, num_epochs: int = 5, run_name: str = None):
    name = run_name or f"lr_{lr:g}"
    writer = SummaryWriter(log_dir=str(RUNS_ROOT / name))
    torch.manual_seed(0)
    model = SmallMLP().to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "val_acc": [], "grad_norm": []}
    for epoch in range(num_epochs):
        model.train()
        running, n_seen, gnorms = 0.0, 0, []
        for x, y in train_loader:
            opt.zero_grad()
            logits = model(x)
            loss   = criterion(logits, y)
            loss.backward()
            gnorms.append(global_grad_norm(model))
            opt.step()
            running += loss.item() * x.size(0); n_seen += x.size(0)
        train_loss = running / n_seen

        model.eval()
        loss_sum, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                logits = model(x)
                loss_sum += criterion(logits, y).item() * x.size(0)
                correct  += (logits.argmax(dim=1) == y).sum().item()
                total    += y.size(0)
        val_loss = loss_sum / total
        val_acc  = correct / total

        mean_gnorm = float(np.mean(gnorms))
        wnorm = global_weight_norm(model)

        writer.add_scalar("loss/train",        train_loss, epoch)
        writer.add_scalar("loss/val",          val_loss,   epoch)
        writer.add_scalar("metric/val_acc",    val_acc,    epoch)
        writer.add_scalar("debug/grad_norm",   mean_gnorm, epoch)
        writer.add_scalar("debug/weight_norm", wnorm,      epoch)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["grad_norm"].append(mean_gnorm)

    writer.close()
    return history

## Run three LRs in a row

Pick three: deliberately too small, sane, deliberately too large. The
goal is to learn what each looks like on the curves.

In [4]:
results = {}
for lr in [1e-4, 1e-2, 1.0]:
    t0 = time.time()
    results[f"lr={lr:g}"] = train_one_lr(lr, num_epochs=5)
    print(f"  lr={lr:g}: done in {time.time() - t0:.1f}s, "
          f"final val_acc={results[f'lr={lr:g}']['val_acc'][-1]:.4f}, "
          f"final grad_norm={results[f'lr={lr:g}']['grad_norm'][-1]:.2e}")

/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  lr=0.0001: done in 2.5s, final val_acc=0.5840, final grad_norm=1.93e+00


  lr=0.01: done in 2.5s, final val_acc=0.9000, final grad_norm=1.17e+00


  lr=1: done in 2.5s, final val_acc=0.0870, final grad_norm=1.38e-01


## Compare the three runs

Plotted inline. Same data is also live in TensorBoard — open it with:

```bash
tensorboard --logdir experiments/lab06/runs
```

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for name, h in results.items():
    axes[0].plot(h["val_loss"],  marker="o", label=name)
    axes[1].plot(h["val_acc"],   marker="o", label=name)
    axes[2].semilogy(h["grad_norm"], marker="o", label=name)
axes[0].set_title("Validation loss");    axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
axes[2].set_title("Gradient norm (log scale)"); axes[2].set_xlabel("epoch"); axes[2].legend()
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

/tmp/ipykernel_510110/2900660157.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**How to read the three curves**

- **`lr=1e-4`** — val loss decreases very slowly, val acc barely above
  chance after 5 epochs. Gradient norm is healthy (no explosion); the
  problem is just that the steps are too small.
- **`lr=1e-2`** — val loss drops smoothly, val acc converges to a
  sensible plateau, gradient norm settles into a steady range. This is
  the *sane* regime.
- **`lr=1.0`** — gradient norm jumps wildly between epochs, val loss
  oscillates or stalls, val acc may not move past chance. This is the
  *too large* regime.

The general lesson from `docs/12_practical_methodology_debugging.md`:
**always log gradient norm**. It is the cheapest scalar that reveals
optimizer pathologies long before they corrupt validation metrics.

## What to add to your own training scripts

```python
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter(log_dir="experiments/<run>/runs")

# Inside the epoch loop:
total = 0.0
for p in model.parameters():
    if p.grad is not None:
        total += p.grad.pow(2).sum().item()
writer.add_scalar("debug/grad_norm", total ** 0.5, epoch)
writer.add_scalar("loss/train", train_loss, epoch)
writer.add_scalar("loss/val",   val_loss,   epoch)
writer.add_scalar("metric/val_acc", val_acc, epoch)

writer.close()
```

Six lines, no other dependency. Re-run with different configs and
TensorBoard overlays the runs automatically.